# einops-repeat — ex5: 2×2 nearest-neighbor upsample (decompose + stretch + compose)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.repeat` patterns that ramp from new-axis broadcast → per-token-to-per-feature → vertical stretch → horizontal tile → 2×2 nearest-neighbor upsample. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-repeat`**, which bridges to the bank subtopic `Einops: Repeat` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'c h w -> b c h w'` with `b=4` broadcasts across a new batch dim.
2. **Trailing axis** — `'b t -> b t d'` with `d=64` materializes a per-token weight at per-feature width.
3. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a block.
4. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two copies of every row.

Difference between **stretch** `(h r)` and **tile** `(r h)`: the factor written first varies slower. `(h r)` puts source row 0 at positions `0..r-1`; `(r h)` puts source row 0 at positions `0, h, 2h, ...`.

### Exercise 5 — 2×2 nearest-neighbor upsample (decompose + stretch + compose)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize new-axis introduction with axis composition to perform 2-D nearest-neighbor upsampling.
> Keywords: upsample, nearest-neighbor, integration, multi-kc
> ```

**KCs targeted:** `repeat-add-axis`, `repeat-stretch-via-composition`, `repeat-nearest-upsample`

Implement `ex5_upsample_2x2(x)` to nearest-neighbor upsample a batch of feature maps by 2× in both spatial dimensions.

Input shape: `(b, c, h, w)`. Output shape: `(b, c, 2h, 2w)`. Each input pixel `x[..., i, j]` should appear as a `2×2` block at output positions `y[..., 2i:2i+2, 2j:2j+2]`.

Introduce two new axes `p1, p2` of size 2, then compose them with the spatial axes so each pixel stretches into a 2×2 block.

Equivalent to `torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')`.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one pattern; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_upsample_2x2(x: Tensor) -> Tensor:
    """Nearest-neighbor 2x2 upsample. (b, c, h, w) → (b, c, 2h, 2w).

    Each input pixel becomes a 2x2 block of identical values.
    """
    raise NotImplementedError()


def _test_ex5():
    x = t.arange(1 * 1 * 2 * 2).reshape(1, 1, 2, 2).float()
    y = ex5_upsample_2x2(x)
    assert y.shape == (1, 1, 4, 4), f'expected (1,1,4,4), got {y.shape}'
    expected = F.interpolate(x, scale_factor=2, mode='nearest')
    assert t.equal(y, expected), 'values differ from F.interpolate(scale=2, mode=nearest)'

    # Also test a larger random tensor.
    x2 = t.randn(2, 3, 5, 7)
    y2 = ex5_upsample_2x2(x2)
    assert y2.shape == (2, 3, 10, 14), f'expected (2,3,10,14), got {y2.shape}'
    assert t.allclose(y2, F.interpolate(x2, scale_factor=2, mode='nearest')), 'random-input mismatch'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_upsample_2x2(x: Tensor) -> Tensor:
    return repeat(
        x,
        'b c h w -> b c (h p1) (w p2)',
        p1=2, p2=2,
    )
```

**Reading the pattern.**
- `(h p1)` on the output side: source row `i` fills output rows `i*p1 .. i*p1+p1-1`. Same shape recipe as Exercise 3, but applied to an axis already present in the input.
- `(w p2)` does the same horizontally.
- `b` and `c` pass through untouched.

**Stretch vs tile here.** `(h p1)` stretches — every source pixel fills a contiguous 2×2 patch. If you wrote `(p1 h)` instead you'd get the entire row tiled twice vertically, which is **not** nearest-neighbor upsampling.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()